# Практика 3. Приведення до tidy data (варіант 3)

**Студент:** Стас Катренко, група ІТ-41  
**Місто:** Одеса  
**Файл даних:** ariant.csv, ariant.json (генеруються під час виконання)

## Мета
- Перетворити wide-таблицю середньомісячної хмарності у tidy-формат через melt().
- Перевірити зворотне перетворення через pivot().
- Зберегти tidy-таблицю у CSV та JSON і перевірити еквівалентність при повторному читанні.
- Відповісти на підсумкові питання про типи шкал і відмінності форматів.

In [ ]:
from IPython.display import display
import pandas as pd

In [5]:
wide = pd.DataFrame(
    {
        "місто": ["Одеса"],
        "Січ": [70],
        "Лют": [66],
        "Бер": [60],
        "Кві": [50],
        "Тра": [42],
        "Чер": [36],
        "Лип": [32],
        "Сер": [34],
        "Вер": [44],
        "Жов": [58],
        "Лис": [68],
        "Гру": [74],
    }
)

print(wide)

   місто  Січ  Лют  Бер  Кві  Тра  Чер  Лип  Сер  Вер  Жов  Лис  Гру
0  Одеса   70   66   60   50   42   36   32   34   44   58   68   74


In [12]:
tidy = wide.melt(id_vars="місто", var_name="місяць", value_name="Хмарність")

print(tidy)
print("\nФорма tody-таблиці (shape):", tidy.shape)

    місто місяць  Хмарність
0   Одеса    Січ         70
1   Одеса    Лют         66
2   Одеса    Бер         60
3   Одеса    Кві         50
4   Одеса    Тра         42
5   Одеса    Чер         36
6   Одеса    Лип         32
7   Одеса    Сер         34
8   Одеса    Вер         44
9   Одеса    Жов         58
10  Одеса    Лис         68
11  Одеса    Гру         74

Форма tody-таблиці (shape): (12, 3)


In [16]:
tidy = wide.melt(id_vars="місто", var_name="місяць", value_name="хмарність")

back = tidy.pivot(
    index="місто", columns="місяць", values="хмарність"
).reset_index()

back.columns.name = None

months_order = [
    "місто",
    "Січ",
    "Лют",
    "Бер",
    "Кві",
    "Тра",
    "Чер",
    "Лип",
    "Сер",
    "Вер",
    "Жов",
    "Лис",
    "Гру",
]
back = back[months_order]

print("Збіг з оригіналом:", wide.equals(back))

Збіг з оригіналом: True


In [17]:
tidy.to_csv("variant.csv", index=False)
tidy.to_json("variant.json", orient="records", force_ascii=False)

resoaded_csv = pd.read_csv("variant.csv")
resoaded_json = pd.read_json("variant.json")

print("CSV збігається:", tidy.equals(resoaded_csv))
print("JSON збігається:", tidy.equals(resoaded_json))

print("\n--- Зміст variant.csv ---")
print(open("variant.csv", encoding="utf-8").read())

print("--- Зміст variant.json ---")
print(open("variant.json", encoding="utf-8").read())

CSV збігається: True
JSON збігається: True

--- Зміст variant.csv ---
місто,місяць,хмарність
Одеса,Січ,70
Одеса,Лют,66
Одеса,Бер,60
Одеса,Кві,50
Одеса,Тра,42
Одеса,Чер,36
Одеса,Лип,32
Одеса,Сер,34
Одеса,Вер,44
Одеса,Жов,58
Одеса,Лис,68
Одеса,Гру,74

--- Зміст variant.json ---
[{"місто":"Одеса","місяць":"Січ","хмарність":70},{"місто":"Одеса","місяць":"Лют","хмарність":66},{"місто":"Одеса","місяць":"Бер","хмарність":60},{"місто":"Одеса","місяць":"Кві","хмарність":50},{"місто":"Одеса","місяць":"Тра","хмарність":42},{"місто":"Одеса","місяць":"Чер","хмарність":36},{"місто":"Одеса","місяць":"Лип","хмарність":32},{"місто":"Одеса","місяць":"Сер","хмарність":34},{"місто":"Одеса","місяць":"Вер","хмарність":44},{"місто":"Одеса","місяць":"Жов","хмарність":58},{"місто":"Одеса","місяць":"Лис","хмарність":68},{"місто":"Одеса","місяць":"Гру","хмарність":74}]


**Відповіді на підсумкові питання**

1. **Чому "місто" — номінальна шкала, а "хмарність" — шкала відношень?**
Стовпець "місто" є номінальним, бо назви міст слугують лише якісними мітками без природного порядку або кількісного виміру. Стовпець "хмарність" належить до шкали відношень, оскільки вимірюється в числових одиницях (%) з наявним фізичним абсолютним нулем (0% = відсутність хмар), що дозволяє порівнювати значення у скільки завгодно разів.
2. **Чому вихідна wide-таблиця не вважається tidy, і що змінюється після melt()?**
Таблиця wide не є tidy, оскільки назви стовпців ("Січ", "Лют" тощо) містять значення змінної "місяць", а не назви окремих параметрів. За правилами Tidy Data, кожна змінна має бути в окремому стовпці, а кожне спостереження — в окремому рядку. Після `melt()` структуризація змінюється: стовпці місяців згортаються у значення одного стовпця "місяць", а відповідні їм показники формують стовпець "хмарність".
3. **Чим формат CSV принципово відрізняється від JSON при зберіганні однієї й тієї самої таблиці?**
CSV описує дані у вигляді плоскої матриці текст/значення через роздільники (коми та переноси рядків) без збереження типів даних. JSON зберігає дані як древоподібну або масивну структуру пар "ключ-значення" з явним збереженням типів (integer, float, string, boolean, null) та можливістю зберігання вкладених об'єктів і списків.